## Import Required Librariesm

In [ ]:
import pandas as pd
import numpy as np
import eda_toolkit as et
from pathlib import Path
import plotly.express as px

import sys
sys.path.append("./")

from core.gis_mapping_functions import *

## Set Paths

In [ ]:
# Current working directory (should be project_root/notebooks)
NOTEBOOK_DIR = Path.cwd()

# Go up one level to project root
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Build path to CSV
FILE_PATH = PROJECT_ROOT / "data" / "viz_data" / "log_preds_merge_r2_38.csv"

# Image path
SVG_PATH = PROJECT_ROOT / "images" / "svg_images"

PNG_PATH = PROJECT_ROOT / "images" / "png_images"

HTML_PATH = PROJECT_ROOT / "images" / "html_images"

print("Loading from:", FILE_PATH)

df = pd.read_csv(FILE_PATH)

print("Shape:", df.shape)

In [ ]:
df[["pred_fatalities", "actual_fatalities"]]

In [ ]:
df.pred_fatalities.sum()

## 1) Validate required columns

In [ ]:
required_cols = [
    "actual_fatalities",
    "pred_fatalities",
    "latitude",
    "longitude",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing}\nAvailable columns: {list(df.columns)}"
    )

# Ensure numeric types
for col in ["actual_fatalities", "pred_fatalities", "latitude", "longitude"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows that can't be mapped
df = df.dropna(
    subset=["latitude", "longitude", "actual_fatalities", "pred_fatalities"]
).copy()

print("Rows after NA drop:", len(df))

## 2) Remove structural zero noise

In [ ]:
# Keep events where either actual OR predicted > 0
df = df[(df["actual_fatalities"] > 0) | (df["pred_fatalities"] > 0)].copy()

print("Rows after zero-zero filter:", len(df))

In [ ]:
df.admin1.value_counts()

In [ ]:
et.dataframe_profiler(df=df.select_dtypes(include=["object"]))

## 2b (Optional) -- Donetsk only

In [ ]:


donetsk_df = df[(df.admin1 == "Donetsk")].copy()

## 3) Create error features

In [ ]:
df["error"] = df["pred_fatalities"] - df["actual_fatalities"]
df["abs_error"] = df["error"].abs()

# Optional: percent error (stable version)
df["pct_error"] = df["error"] / (df["actual_fatalities"] + 1)

print("Error summary:")
print(df["error"].describe())

## 4) Feature engineering for mapping

In [ ]:
df["error"] = df["pred_fatalities"] - df["actual_fatalities"]
df["abs_error"] = df["error"].abs()

# Helpful for skewed casualty distributions (optional)
df["pct_error"] = df["error"] / (df["actual_fatalities"] + 1)

# Optional: log-space error (if you have log_pred_fatalities already)
if "log_pred_fatalities" in df.columns:
    df["log_pred_fatalities"] = pd.to_numeric(
        df["log_pred_fatalities"], errors="coerce"
    )
    df["log_actual_fatalities"] = np.log1p(df["actual_fatalities"])
    df["log_error"] = df["log_pred_fatalities"] - df["log_actual_fatalities"]

## 5) Plotly scatter map

In [ ]:
hover_cols = []
for c in [
    "actual_fatalities",
    "pred_fatalities",
    "error",
    "abs_error",
    "event_group",
    "admin1",
    "interaction",
]:
    if c in df.columns:
        hover_cols.append(c)

fig = px.scatter_mapbox(
    df,
    lat="latitude",
    lon="longitude",
    color="error",
    size="abs_error",
    size_max=18,
    hover_data=hover_cols,
    color_continuous_scale="Spectral",
    range_color=(-20, 10),
    mapbox_style="carto-positron",
    zoom=5,
    center={"lat": 49, "lon": 31},
)

fig.update_layout(
    title="Predicted vs Actual Fatalities (Residual Map)",
    paper_bgcolor="white",
    plot_bgcolor="white",
    mapbox=dict(style="carto-positron"),
    coloraxis_colorbar=dict(
        title="Error (Pred - Actual)",
        bgcolor="rgba(240,240,240,1)",
        bordercolor="black",
        borderwidth=1,
    ),
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

## 6) Export options

In [ ]:
# --- Option A: High-res PNG (recommended for publications) ---
# pip install kaleido
fig.write_image(PNG_PATH / "residual_map.png", width=1400, height=900, scale=2)

# --- Option B: Interactive HTML ---
fig.write_html(HTML_PATH / "residual_map.html", include_plotlyjs="cdn")

# --- Option C: SVG via scatter_geo (no tile layer, vector-safe) ---
fig_geo = px.scatter_geo(
    df,
    lat="latitude",
    lon="longitude",
    color="error",
    size="abs_error",
    size_max=18,
    hover_data=hover_cols,
    color_continuous_scale="Spectral",
    range_color=(-20, 10),
    projection="natural earth",
)

fig_geo.update_layout(
    title="Predicted vs Actual Fatalities (Residual Map)",
    paper_bgcolor="white",
    plot_bgcolor="white",
    coloraxis_colorbar=dict(
        title="Error (Pred - Actual)",
        bgcolor="rgba(240,240,240,1)",
        bordercolor="black",
        borderwidth=1,
    ),
    geo=dict(
        showland=True,
        landcolor="lightgray",
        showocean=True,
        oceancolor="lightblue",
        showcoastlines=True,
        coastlinecolor="white",
        showcountries=True,  # replaces showborders
        countrycolor="white",  # replaces bordercolor
        fitbounds="locations",
        resolution=50,
    ),
    margin=dict(l=0, r=0, t=40, b=0),
)

fig_geo.show()
fig_geo.write_image(SVG_PATH / "residual_map.svg")

In [ ]:
# By event_group
# The top three event_group values are represented, all others are represented as "Other"

major_types = ["Armed clash", "Air/drone strike", "Shelling/artillery/missile attack"]

df["event_group"] = df["sub_event_type"].apply(
    lambda x: x if x in major_types else "Other"
)

hover_cols = [
    "actual_fatalities",
    "pred_fatalities",
    "error",
    "abs_error",
    "admin1",
    "event_group",
]

for event in sorted(df["event_group"].unique()):
    subset = df[df["event_group"] == event].copy()

    fig = px.scatter_mapbox(
        subset,
        lat="latitude",
        lon="longitude",
        color="error",
        size="abs_error",
        size_max=18,
        hover_data=hover_cols,
        color_continuous_scale="Spectral",
        range_color=(-20, 20),
        mapbox_style="carto-positron",
        zoom=5,
        center={"lat": 49, "lon": 31},
        title=f"Residual Map: {event}",
    )

    fig.update_layout(
        paper_bgcolor="white",
        plot_bgcolor="white",
        coloraxis_colorbar=dict(
            title="Error (Pred - Actual)",
            bgcolor="rgba(240,240,240,1)",
            bordercolor="black",
            borderwidth=1,
        ),
        margin=dict(l=0, r=0, t=40, b=0),
    )

    fig.show()

## Static Matplotlib GeoPandas Figures

In [ ]:
full_vmax = float(np.ceil(np.percentile(df["error"].abs(), 99) / 5) * 5)

### All events, Donetsk-zoomed

In [ ]:
# All events, Donetsk-zoomed
df_all = df.copy()
df_all["all_events"] = "all"  # dummy column so the function has something to filter on
plot_residual_map(
    df_all,
    "all",
    group_col="all_events",
    clip_to_oblasts=["Donets'k"],
    title="Spatial distribution of residuals across Donetsk oblast",
    save=True,
    title_fontsize=11,
    label_fontsize=9,
    map_fontsize=10,
)

### Air/drone strike

In [ ]:
plot_residual_map(
    df, "Air/drone strike",
    clip_to_oblasts=["Donets'k"],
    title="Spatial distribution of prediction residuals across Donetsk oblast: Air/drone strike",
    save=True,
    # vmin=-full_vmax, vmax=full_vmax,
    title_fontsize=11,
    label_fontsize=9,
    map_fontsize=10,
)

### Shelling/artillery/missile attack

In [ ]:
plot_residual_map(
    df, "Shelling/artillery/missile attack",
    clip_to_oblasts=["Donets'k"],
    title="Spatial distribution of prediction residuals across Donetsk oblast: Shelling/artillery/missile attack",
    save=True,
    # vmin=-full_vmax, vmax=full_vmax,
    title_fontsize=11,
    label_fontsize=9,
    map_fontsize=10,
)

### Armed clash

In [ ]:
plot_residual_map(
    df, "Armed clash",
    cities="auto",
    title="Spatial distribution of prediction residuals across Ukraine: Armed clash",
    save=True,
    # vmin=-full_vmax, vmax=full_vmax,
    title_fontsize=11,
    label_fontsize=9,
    map_fontsize=10,
    
)

### Multiple calls with a consistent scale across all of them

In [ ]:
# (compute vmax once on full data, pass through)
full_vmax = float(np.ceil(np.percentile(df["error"].abs(), 99) / 5) * 5)
for et in ["Air/drone strike", "Armed clash", "Shelling/artillery/missile attack"]:
    plot_residual_map(df, et, vmin=-full_vmax, vmax=full_vmax)

# Use the collapsed event_group column instead of sub_event_type
plot_residual_map(df, "Other", group_col="event_group")

# Custom title, skip save and show (useful when embedding in another figure)
plot_residual_map(
    df,
    "Air/drone strike",
    title="Air/drone strike residuals — XGBoost test set",
    save=False,
)